In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
)

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    logging as hf_logging,        
)

import evaluate                        
from sentence_transformers import (
    SentenceTransformer,
    util as st_util,
)

In [94]:
# Load the passages split
passages = load_dataset("rag-datasets/rag-mini-bioasq", "text-corpus", split="passages")

# Load the test set (QA format)
test = load_dataset("rag-datasets/rag-mini-bioasq", "question-answer-passages", split="test")

# Optional: convert to pandas
passages_df = passages.to_pandas()
test_df = test.to_pandas()

In [ ]:

# ------------------------------------------------------------------
# df is your FULL dataframe. Make sure it has at least 1 500 rows!
# ------------------------------------------------------------------
df = test_df                                # rename as needed

# 1️⃣  Take a reproducible 1 500‑row sample up front
base_df = df.sample(n=1_500, random_state=42)

# 2️⃣  First split → fixed 500‑row training set
train_df, remain_df = train_test_split(
    base_df,
    train_size=500,                 # absolute count
    random_state=42,
)

# 3️⃣  Second split → 500‑row validation & 500‑row test
val_df, test_df = train_test_split(
    remain_df,
    train_size=500,                 # half of 1 000
    random_state=42,
)

print(f"Train: {len(train_df)}")      # 500
print(f"Val:   {len(val_df)}")        # 500
print(f"Test:  {len(test_df)}")       # 500


Train: 500
Val:   500
Test:  500


In [96]:
passages_df.head()

,passage,id
0,New data on viruses isolated from patients wit...,9797
1,We describe an improved method for detecting d...,11906
2,We have studied the effects of curare on respo...,16083
3,Kinetic and electrophoretic properties of 230-...,23188
4,Male Wistar specific-pathogen-free rats aged 2...,23469


In [97]:
train_df.head()

,question,answer,relevant_passage_ids,id
3064,What is the mechanism of the drug CRT0066101?,Recently developed small molecule PKD inhibito...,"[26797128, 26895471, 20947701, 29270134, 29071...",3064
3008,Can Diazepam be beneficial in the treatment o...,Diazepam treatment improved cognitive recovery...,"[8877308, 10760494]",3008
665,Has overexpression of sirtuins been reported t...,Overexpression of sirtuins (NAD(+)-dependent p...,"[23082874, 12006491, 21938067, 15308206]",665
3803,Is Tcf3 associated with the Wnt pathway?,Tcf3 is a component of the Wnt/β-catenin and N...,"[31033094, 23505158, 19074834, 24832538, 25832...",3803
2566,What is the role of tankyrases in response to ...,Tankyrases promote homologous recombination an...,[26845027],2566


In [98]:
true_false_examples = train_df[train_df['question'].str.lower().str.startswith("is")]

In [99]:
true_false_examples.head()

,question,answer,relevant_passage_ids,id
3803,Is Tcf3 associated with the Wnt pathway?,Tcf3 is a component of the Wnt/β-catenin and N...,"[31033094, 23505158, 19074834, 24832538, 25832...",3803
3899,Is erabutoxin b usually found in plants?,Erabutoxin b is a short-chain neurotoxic pepti...,"[7407041, 2514275, 4664580, 6279398, 7526378, ...",3899
4273,Is tofacitinib a JAK inhibitor?,"Yes, tofacitinib is a small JAK inhibitor.",[31721311],4273
309,Is TREM2 associated with Alzheimer's disease i...,TREM2 variants have been found to be associate...,"[23391427, 17088018, 23150908, 23150934, 23407...",309
3555,Is AZD5153 active in prostate cancer?,"Yes, AZD5153 was shown to be effective in trea...",[30308485],3555


In [100]:
factoid_examples = train_df[~train_df['question'].str.lower().str.startswith("is")]

In [101]:
factoid_examples.head()

,question,answer,relevant_passage_ids,id
3064,What is the mechanism of the drug CRT0066101?,Recently developed small molecule PKD inhibito...,"[26797128, 26895471, 20947701, 29270134, 29071...",3064
3008,Can Diazepam be beneficial in the treatment o...,Diazepam treatment improved cognitive recovery...,"[8877308, 10760494]",3008
665,Has overexpression of sirtuins been reported t...,Overexpression of sirtuins (NAD(+)-dependent p...,"[23082874, 12006491, 21938067, 15308206]",665
2566,What is the role of tankyrases in response to ...,Tankyrases promote homologous recombination an...,[26845027],2566
657,List all reported treatment options for anxiet...,The predominant approach is to use versions of...,"[22964266, 20694508, 18437549, 23118256, 21571...",657


## We’ll extract 2 True/False and 2 Factoid examples for Few-Shot prompts

In [102]:
few_shot_examples = pd.concat([true_false_examples.head(2), factoid_examples.head(2)])


In [103]:
few_shot_examples.head()

,question,answer,relevant_passage_ids,id
3803,Is Tcf3 associated with the Wnt pathway?,Tcf3 is a component of the Wnt/β-catenin and N...,"[31033094, 23505158, 19074834, 24832538, 25832...",3803
3899,Is erabutoxin b usually found in plants?,Erabutoxin b is a short-chain neurotoxic pepti...,"[7407041, 2514275, 4664580, 6279398, 7526378, ...",3899
3064,What is the mechanism of the drug CRT0066101?,Recently developed small molecule PKD inhibito...,"[26797128, 26895471, 20947701, 29270134, 29071...",3064
3008,Can Diazepam be beneficial in the treatment o...,Diazepam treatment improved cognitive recovery...,"[8877308, 10760494]",3008


## Format Prompt for Few-Shot Learning

In [ ]:

def get_passage_text_by_id(passage_ids, passages_df, max_passages=1):
    # Ensure passage_ids is a list
    if isinstance(passage_ids, str):
        try:
            passage_ids = ast.literal_eval(passage_ids)
        except (ValueError, SyntaxError):
            return "Invalid passage ID format."
    
    # Convert passage_ids to a set of integers if needed
    try:
        passage_ids = [int(pid) for pid in passage_ids]
    except (ValueError, TypeError):
        return "Passage IDs must be integers."

    # Ensure passages_df index is integer type
    if not pd.api.types.is_integer_dtype(passages_df.index):
        passages_df.index = passages_df.index.astype(int)

    # Filter only valid IDs
    valid_ids = [pid for pid in passage_ids if pid in passages_df.index]
    if not valid_ids:
        return "No relevant context found."

    # Retrieve passages
    selected_passages = passages_df.loc[valid_ids]['passage'].head(max_passages).tolist()
    return "\n".join(selected_passages)


In [105]:
def build_prompt_with_context(examples, test_question, test_relevant_passage_ids, passages_df, use_chain_of_thought=False):
    """
    Builds a prompt with dynamic Chain-of-Thought based on question length
    Returns: (prompt_text, original_question) tuple
    """
    # Dynamic CoT triggering
    use_cot = use_chain_of_thought or len(test_question.split()) > 10
    
    # Core instruction
    instruction = ("You are a biomedical QA assistant. "
                  "Answer concisely and only when certain.\n")
    
    if use_cot:
        instruction += ("When answering complex questions, "
                       "think step by step:\n"
                       "1. Identify key concepts\n"
                       "2. Analyze the context\n"
                       "3. Synthesize information\n"
                       "4. Provide final answer\n\n")
    else:
        instruction += "\n"
    
    # Add few-shot examples (with CoT when applicable)
    for _, row in examples.iterrows():
        example_cot = len(row['question'].split()) > 10
        instruction += f"Question: {row['question']}\n"
        if example_cot:
            instruction += ("Reasoning: [Example reasoning steps]\n"
                          f"Answer: {row['answer']}\n\n")
        else:
            instruction += f"Answer: {row['answer']}\n\n"
    
    # Get context for current question
    context = get_passage_text_by_id(test_relevant_passage_ids, passages_df)
    
    # Build final prompt
    prompt = (
        f"{instruction}"
        f"Context: {context}\n"
        f"Question: {test_question}\n"
        f"{'Reasoning: ' if use_cot else ''}"  # Add reasoning line if CoT
    )
    
    return prompt

In [106]:
# Dynamic CoT toggle based on question length
sample_row = train_df.iloc[252]
question = sample_row['question']
use_cot = len(question.split()) > 10  # or any heuristic you prefer

prompt = build_prompt_with_context(
    examples=few_shot_examples,
    test_question=question,
    test_relevant_passage_ids=sample_row['relevant_passage_ids'],
    passages_df=passages_df,
    use_chain_of_thought=use_cot
)

print(prompt)


You are a biomedical QA assistant. Answer concisely and only when certain.
When answering complex questions, think step by step:
1. Identify key concepts
2. Analyze the context
3. Synthesize information
4. Provide final answer

Question: Is Tcf3 associated with the Wnt pathway?
Answer: Tcf3 is a component of the Wnt/β-catenin and Notch signaling pathways.

Question: Is erabutoxin b usually found in plants?
Answer: Erabutoxin b is a short-chain neurotoxic peptide purified from the venom of the sea snake Laticauda semifasciata.

Question: What is the mechanism of the drug CRT0066101?
Answer: Recently developed small molecule PKD inhibitors, CID755673 and CRT0066101, provide potentially important pharmacological approaches to further investigate the effect of PKD in pancreatitis therapy

Question: Can Diazepam be beneficial  in the treatment of  traumatic brain injury?
Reasoning: [Example reasoning steps]
Answer: Diazepam treatment improved cognitive recovery and mortality in brain injure

In [107]:
# Load FLAN-T5-base
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [108]:

# Tokenize prompt
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,    # allow some randomness
    temperature=0.7,   # balance creativity and accuracy
    top_p=0.9,         # nucleus sampling
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Decode and print
decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("Model Output:\n", decoded_output.strip())


Model Output:
 nintedanib is a drug that can treat lung disease associated with systemic sclerosis.


## Now that we can see that our model is working as intended with created prompt, we will now move on to model training 

## first lets create our train, val andtest dataest as same as prompt 

In [109]:
def create_dataset(df, passages_df, few_shot_examples, tokenizer):
    inputs = []
    targets = []
    
    for _, row in df.iterrows():
        # Build prompt
        prompt = build_prompt_with_context(
            examples=few_shot_examples,
            test_question=row['question'],
            test_relevant_passage_ids=row['relevant_passage_ids'],
            passages_df=passages_df
        )
        
        inputs.append(prompt)
        targets.append(row['answer'])
    
    # Tokenize with padding
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        model_inputs["labels"] = labels["input_ids"]
        
    return Dataset.from_dict(model_inputs)

In [110]:
train_dataset= create_dataset(train_df, passages_df, few_shot_examples,tokenizer)
val_dataset = create_dataset(val_df, passages_df, few_shot_examples,tokenizer)
test_dataset = create_dataset(test_df, passages_df, few_shot_examples,tokenizer)

In [111]:
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 500
Validation samples: 500
Test samples: 500


In [112]:
# Verify question-answer pairs before training
for i in range(3):
    print(f"Q: {train_df.iloc[i]['question']}")
    print(f"A: {train_df.iloc[i]['answer']}\n")

Q: What is the mechanism of the drug CRT0066101?
A: Recently developed small molecule PKD inhibitors, CID755673 and CRT0066101, provide potentially important pharmacological approaches to further investigate the effect of PKD in pancreatitis therapy

Q: Can Diazepam be beneficial  in the treatment of  traumatic brain injury?
A: Diazepam treatment improved cognitive recovery and mortality in brain injured rats.

Q: Has overexpression of sirtuins been reported to increase lifespan in budding yeast (Saccharomyces cerevisiae)?
A: Overexpression of sirtuins (NAD(+)-dependent protein deacetylases) has been reported to increase lifespan in budding yeast (Saccharomyces cerevisiae).



In [113]:
# Check first 3 samples
for i in range(3):
    sample = train_dataset[i]  # Use integer indexing
    print(f"\nSample {i}:")
    print("Input:", tokenizer.decode(sample['input_ids'], skip_special_tokens=True))
    print("Label:", tokenizer.decode(sample['labels'], skip_special_tokens=True))
    print("Input IDs shape:", sample['input_ids'].shape if hasattr(sample['input_ids'], 'shape') else len(sample['input_ids']))


Sample 0:
Input: You are a biomedical QA assistant. Answer concisely and only when certain. Question: Is Tcf3 associated with the Wnt pathway? Answer: Tcf3 is a component of the Wnt/-catenin and Notch signaling pathways. Question: Is erabutoxin b usually found in plants? Answer: Erabutoxin b is a short-chain neurotoxic peptide purified from the venom of the sea snake Laticauda semifasciata. Question: What is the mechanism of the drug CRT0066101? Answer: Recently developed small molecule PKD inhibitors, CID755673 and CRT0066101, provide potentially important pharmacological approaches to further investigate the effect of PKD in pancreatitis therapy Question: Can Diazepam be beneficial in the treatment of traumatic brain injury? Reasoning: [Example reasoning steps] Answer: Diazepam treatment improved cognitive recovery and mortality in brain injured rats. Context: No relevant context found. Question: What is the mechanism of the drug CRT0066101? 
Label: Recently developed small molecule

In [115]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [116]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-bioqa",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=50,     # works even in old versions
    do_train=True,
    do_eval=True,
)

# ----- manually add the modern attributes -----
training_args.evaluation_strategy = "steps"
training_args.save_strategy       = "epoch"
training_args.logging_strategy    = "steps"
training_args.logging_first_step  = True
training_args.predict_with_generate = True
# ----------------------------------------------

# now pass training_args to Seq2SeqTrainer as usual


In [118]:
# 8. Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=tokenizer.pad_token_id,
    pad_to_multiple_of=8 if training_args.fp16 else None
)

In [119]:
rouge = evaluate.load("rouge")
    
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v * 100, 4) for k, v in result.items()}
    
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\amrut\AppData\Local\Temp\ipykernel_6848\190236275.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [120]:
# 9. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

C:\Users\amrut\AppData\Local\Temp\ipykernel_6848\1930038966.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [121]:
print(device)

cuda


In [122]:
# Start training
trainer.train()
trainer.save_model("best_flant5_bioqa")

{'loss': 36.5345, 'grad_norm': 269.2979736328125, 'learning_rate': 3e-05, 'epoch': 0.008}
{'loss': 14.9679, 'grad_norm': 30.991134643554688, 'learning_rate': 2.608e-05, 'epoch': 0.4}
{'loss': 4.0248, 'grad_norm': 14.344554901123047, 'learning_rate': 2.208e-05, 'epoch': 0.8}
{'loss': 2.8829, 'grad_norm': 9.470368385314941, 'learning_rate': 1.808e-05, 'epoch': 1.2}
{'loss': 2.1382, 'grad_norm': 7.283683776855469, 'learning_rate': 1.408e-05, 'epoch': 1.6}
{'loss': 1.825, 'grad_norm': 5.020421504974365, 'learning_rate': 1.008e-05, 'epoch': 2.0}
{'loss': 1.8072, 'grad_norm': 5.599419116973877, 'learning_rate': 6.08e-06, 'epoch': 2.4}
{'loss': 1.5469, 'grad_norm': 6.255026340484619, 'learning_rate': 2.08e-06, 'epoch': 2.8}
{'train_runtime': 4139.6324, 'train_samples_per_second': 0.362, 'train_steps_per_second': 0.091, 'train_loss': 4.048639475504557, 'epoch': 3.0}


In [ ]:

idef evaluate_qa_model(model, tokenizer, test_dataset, device='cuda'):
    predictions = []
    references = []
    
    model.eval()
    with torch.no_grad():
        for example in tqdm(test_dataset):
            input_ids = torch.tensor(example['input_ids']).unsqueeze(0).to(device)
            attention_mask = torch.tensor(example['attention_mask']).unsqueeze(0).to(device)

            inputs = {
                'input_ids': input_ids,
                'attention_mask': attention_mask
            }
                 
            # Generate prediction
            output = model.generate(**inputs, max_length=150)
            pred = tokenizer.decode(output[0], skip_special_tokens=True)
            predictions.append(pred)
            
            # Decode reference
            labels = torch.tensor(example['labels'])
            labels = torch.where(labels != -100, labels, torch.tensor(tokenizer.pad_token_id))
            ref = tokenizer.decode(labels.tolist(), skip_special_tokens=True)
            references.append(ref)
    
    # Compute ROUGE
    rouge = evaluate.load('rouge')
    rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
    
    # Compute BERTScore
    bertscore = evaluate.load('bertscore')
    bert_scores = bertscore.compute(predictions=predictions, references=references, lang='en')
    bert_avg_scores = {
        'precision': sum(bert_scores['precision']) / len(bert_scores['precision']),
        'recall': sum(bert_scores['recall']) / len(bert_scores['recall']),
        'f1': sum(bert_scores['f1']) / len(bert_scores['f1']),
    }

    return {
        'rouge': rouge_scores,
        'bertscore': bert_avg_scores
    }


In [125]:
results = evaluate_qa_model(model, tokenizer, test_dataset)


100%|██████████| 500/500 [15:35<00:00,  1.87s/it]


In [126]:
print(results)

{'rouge': {'rouge1': np.float64(0.24172265585057134), 'rouge2': np.float64(0.09579543717441386), 'rougeL': np.float64(0.20519670354843128), 'rougeLsum': np.float64(0.2044979165574752)}, 'bertscore': {'precision': 0.8628068822622299, 'recall': 0.8433612691164016, 'f1': 0.8524440224170685}}


In [135]:
# ─────────  BASELINE  (no OOS guard) ─────────
# If you overwrote `model` during Tuning 1, just reload the original:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

baseline_name = "google/flan-t5-base"          # your starting checkpoint
tok_base      = AutoTokenizer.from_pretrained(baseline_name)
model_base    = AutoModelForSeq2SeqLM.from_pretrained(baseline_name).to(device)

oos_questions = [
    "Who won the FIFA World Cup in 2010?",
    "What is the capital city of Australia?",
    "Explain Einstein's theory of relativity in simple terms.",
    "Who painted the Mona Lisa?",
    "What is the tallest mountain in the world?"
]

print("🟡  Baseline answers (no refusal logic):\n")
for q in oos_questions:
    # simplest prompt: just the question
    inp = tok_base(q, return_tensors="pt").to(device)
    out = model_base.generate(**inp, max_new_tokens=60)
    ans = tok_base.decode(out[0], skip_special_tokens=True)
    print(f"Q: {q}\nA: {ans}\n")


🟡  Baseline answers (no refusal logic):

Q: Who won the FIFA World Cup in 2010?
A: switzerland

Q: What is the capital city of Australia?
A: melbourne

Q: Explain Einstein's theory of relativity in simple terms.
A: Einstein's theory of relativity is that the universe is a non-relativistic universe.

Q: Who painted the Mona Lisa?
A: franz heidegger

Q: What is the tallest mountain in the world?
A: sahara



# Tuning 1

| Area                                                                                                                          | Baseline model                          | **Tuning 1 upgrades**                                                                   |
| ----------------------------------------------------------------------------------------------------------------------------- | --------------------------------------- | --------------------------------------------------------------------------------------- |
| **Prompt structure**                                                                                                          | *Few‑Shot (4) + optional CoT*           | • Added **1 refusal exemplar** (`"Who won the FIFA World Cup in 2010?" → OUT_OF_SCOPE`) |
| **Out‑of‑scope guard**                                                                                                        | None (model tried to answer everything) | **Dual guard**                                                                          |


## Add refusal example

In [127]:
refusal_example = pd.DataFrame({
    "question": ["Who won the FIFA World Cup in 2010?"],   # clearly OOD
    "answer":   ["OUT_OF_SCOPE"],
    "relevant_passage_ids": [[]]
})
few_shot_examples = pd.concat([few_shot_examples, refusal_example], ignore_index=True)


## Semantic OOS detector (SBERT gate)

In [ ]:


embedder = SentenceTransformer("all-MiniLM-L6-v2")

# store reference vectors as a **tensor** for fast cosine
ref_vecs = embedder.encode(
    pd.concat([train_df["question"],
               val_df["question"],
               test_df["question"]]).tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True
)                                             # shape: (N, hidden)

def is_in_domain(question: str, tau: float = 0.55) -> bool:
    """True if max cosine‑sim to any BioASQ question ≥ tau."""
    q_vec = embedder.encode([question],        # shape: (1, hidden)
                            convert_to_tensor=True,
                            normalize_embeddings=True)
    sim = util.cos_sim(q_vec, ref_vecs).max().item()  # cosine matrix → max
    return sim >= tau


## Prompt builder (explicit refusal instruction)

In [129]:
def build_prompt_with_context(examples,
                              test_question,
                              test_relevant_passage_ids,
                              passages_df,
                              use_chain_of_thought=False):
    """Return a few‑shot + context prompt with optional CoT header."""
    use_cot = use_chain_of_thought or len(test_question.split()) > 10

    instruction = (
        "You are a biomedical QA assistant. "
        "If a question is outside biomedical science or context is missing, "
        "reply exactly with OUT_OF_SCOPE.\n"
    )

    if use_cot:
        instruction += (
            "When answering complex questions, think step by step:\n"
            "1. Identify key concepts\n"
            "2. Analyze the context\n"
            "3. Synthesize information\n"
            "4. Provide final answer\n\n"
        )
    else:
        instruction += "\n"

    # Few‑shot exemplars (including the new refusal pattern)
    for _, row in examples.iterrows():
        add_cot = len(row["question"].split()) > 10
        instruction += f"Question: {row['question']}\n"
        if add_cot:
            instruction += ("Reasoning: [Example reasoning steps]\n"
                            f"Answer: {row['answer']}\n\n")
        else:
            instruction += f"Answer: {row['answer']}\n\n"

    # Retrieve context passages
    context = get_passage_text_by_id(test_relevant_passage_ids, passages_df)

    return (
        f"{instruction}"
        f"Context: {context}\n"
        f"Question: {test_question}\n"
        f"{'Reasoning: ' if use_cot else ''}"
    )

## Helper for generation with OOS gate

In [130]:
def answer_question(question,
                    passage_ids,
                    passages_df,
                    examples=few_shot_examples,
                    tau=0.55,
                    n_passages=3):             # ← new: fetch k passages
    """
    Return model answer or 'OUT_OF_SCOPE'.
    If passage_ids is empty, automatically grab the k most similar passages.
    """
    # ➊ OOS Gate
    if not is_in_domain(question, tau=tau):
        return "OUT_OF_SCOPE"

    # ➋ Auto‑retrieve context if caller passed [] or None
    if not passage_ids:
        # simple TF‑IDF similarity over passage text (fast & dependency‑free)
        corpus = passages_df["passage"].tolist()
        from sklearn.feature_extraction.text import TfidfVectorizer
        import numpy as np, re

        tfidf = TfidfVectorizer(stop_words="english").fit_transform(
            [question] + corpus
        )
        sims  = (tfidf[0] @ tfidf[1:].T).toarray()[0]
        passage_ids = np.argsort(-sims)[:n_passages].tolist()

    # ➌ Build prompt with the (now valid) passages
    prompt = build_prompt_with_context(
        examples=examples,
        test_question=question,
        test_relevant_passage_ids=passage_ids,
        passages_df=passages_df,
        use_chain_of_thought=False
    )

    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=1024).to(device)

    # ➍ Deterministic decoding
    outs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,          # ← OFF
        temperature=0.0,
        top_p=1.0,
        num_beams=4,              # small beam search
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(outs[0], skip_special_tokens=True).strip()


In [131]:
from tqdm.auto import tqdm
import evaluate, numpy as np, torch

def evaluate_qa_model_oos(model, tokenizer, dataset, passages_df,
                          tau=0.55, device=device):
    preds, refs = [], []

    rouge = evaluate.load("rouge")
    bert  = evaluate.load("bertscore")

    for ex in tqdm(dataset, desc="Evaluating"):
        # Reconstruct plain question text
        question_txt = tokenizer.decode(ex["input_ids"],
                                        skip_special_tokens=True)
        question_txt = question_txt.split("Question:")[-1].strip()

        # Gate check
        if not is_in_domain(question_txt, tau):
            preds.append("OUT_OF_SCOPE")
            refs.append("OUT_OF_SCOPE")
            continue

        # Otherwise generate as usual
        inp_ids = torch.tensor(ex["input_ids"]).unsqueeze(0).to(device)
        attn    = torch.tensor(ex["attention_mask"]).unsqueeze(0).to(device)
        outs = model.generate(
            input_ids=inp_ids,
            attention_mask=attn,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        preds.append(tokenizer.decode(outs[0], skip_special_tokens=True))

        lbl = torch.tensor(ex["labels"])
        lbl = torch.where(lbl != -100, lbl, tokenizer.pad_token_id)
        refs.append(tokenizer.decode(lbl.tolist(), skip_special_tokens=True))

    rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    bs           = bert.compute(predictions=preds, references=refs, lang="en")
    bert_avg     = {k: float(np.mean(bs[k])) for k in ("precision", "recall", "f1")}

    return {"rouge": rouge_scores, "bertscore": bert_avg}

## Evaluation Cell

In [ ]:

small_test_dataset = test_dataset.shuffle(seed=42).select(range(25))

oos_qs = [
    "Who won the FIFA World Cup in 2010?",
    "What is the capital city of Australia?",
    "Explain Einstein's theory of relativity in simple terms.",
    "Who painted the Mona Lisa?",
    "What is the tallest mountain in the world?",
    "What is the capital of India?",
    "Who is the world's best Cricket team?",
    "Who invented telephones?",
    "When was an electric bulb invented?",
    "What is the latest android os?"
]
oos_df = pd.DataFrame({
    "question": oos_qs,
    "answer":   ["OUT_OF_SCOPE"] * len(oos_qs),
    "relevant_passage_ids": [[]] * len(oos_qs)
})

def full_eval(model, tokenizer, id_dataset, oos_df, passages_df, device=device):
    rouge = evaluate.load("rouge")
    bert  = evaluate.load("bertscore")

    id_preds, id_refs = [], []
    oos_preds, oos_refs = [], []

    for ex in tqdm(id_dataset, desc="In‑domain"):
        q_txt = tokenizer.decode(ex["input_ids"], skip_special_tokens=True)
        q_txt = q_txt.split("Question:")[-1].strip()

        lbl_ids = torch.tensor(ex["labels"])
        lbl_ids = torch.where(lbl_ids != -100, lbl_ids,
                              torch.tensor(tokenizer.pad_token_id))
        ref = tokenizer.decode(lbl_ids.tolist(), skip_special_tokens=True)

        pred = answer_question(q_txt, [], passages_df)   # auto‑retrieval
        id_preds.append(pred)
        id_refs.append(ref)

    for _, row in tqdm(oos_df.iterrows(), total=len(oos_df), desc="OOS"):
        pred = answer_question(row["question"], [], passages_df)
        oos_preds.append(pred)
        oos_refs.append("OUT_OF_SCOPE")

    rouge_scores = rouge.compute(predictions=id_preds, references=id_refs, use_stemmer=True)
    bert_scores  = bert.compute(predictions=id_preds, references=id_refs, lang="en")
    bert_avg     = {k: round(float(np.mean(bert_scores[k])), 4)
                    for k in ("precision", "recall", "f1")}

    y_true = [1 if r == "OUT_OF_SCOPE" else 0 for r in (id_refs + oos_refs)]
    y_pred = [1 if p == "OUT_OF_SCOPE" else 0 for p in (id_preds + oos_preds)]

    acc, p, r, f1, _ = accuracy_score(y_true, y_pred), *precision_recall_fscore_support(
        y_true, y_pred, pos_label=1, average="binary", zero_division=0
    )

    return {
        "rouge":       {k: round(v, 4) for k, v in rouge_scores.items()},
        "bertscore":   bert_avg,
        "oos_metrics": {
            "accuracy":  round(acc, 4),
            "precision": round(p, 4),
            "recall":    round(r, 4),
            "f1":        round(f1, 4),
            "total_OOS": len(oos_df),
            "total_ID":  len(id_dataset)
        }
    }

results = full_eval(model, tokenizer, small_test_dataset, oos_df, passages_df)


OOS: 100%|██████████| 10/10 [00:00<00:00, 80.57it/s]


In [133]:
print(results) 

{'rouge': {'rouge1': np.float64(0.2722), 'rouge2': np.float64(0.0964), 'rougeL': np.float64(0.2024), 'rougeLsum': np.float64(0.2013)}, 'bertscore': {'precision': 0.8596, 'recall': 0.865, 'f1': 0.8617}, 'oos_metrics': {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'total_OOS': 10, 'total_ID': 25}}


In [ ]:
def ask_t1(q: str) -> str:
    """Wrapper to call Tuning‑1 answer function."""
    return answer_question(q, [], passages_df)   # [] → auto‑retrieval inside

print("🟢  Tuning‑1 model answers (should refuse OOS):\n")
for q in [
    "Who won the FIFA World Cup in 2010?",
    "What is the capital city of Australia?",
    "Explain Einstein's theory of relativity in simple terms.",
    "Who painted the Mona Lisa?",
    "What is the tallest mountain in the world?"
]:
    print(f"Q: {q}\nA: {ask_t1(q)}\n")


🟢  Tuning‑1 model answers (should refuse OOS):

Q: Who won the FIFA World Cup in 2010?
A: OUT_OF_SCOPE

Q: What is the capital city of Australia?
A: OUT_OF_SCOPE

Q: Explain Einstein's theory of relativity in simple terms.
A: OUT_OF_SCOPE

Q: Who painted the Mona Lisa?
A: OUT_OF_SCOPE

Q: What is the tallest mountain in the world?
A: OUT_OF_SCOPE



## Interpretability

In [141]:
def explain_question(q: str, k_passages: int = 3):
    """Return (answer, reasoning, passages) using Tuning‑1 logic."""
    if not is_in_domain(q):
        return REFUSAL_TEXT, "OOS‑gate triggered", []

    passages = bm25_passages(q, k_passages)
    prompt = (
        "You are a biomedical QA assistant. "
        f"If outside biomedical science, reply exactly with:\n\"{REFUSAL_TEXT}\"\n\n"
    )
    # Few‑Shot block (reuse first 4 examples)
    for _, row in few_shot_examples.head(4).iterrows():
        prompt += f"Question: {row['question']}\nAnswer: {row['answer']}\n\n"

    ctx = "\n".join(passages) if passages else "No relevant context."
    prompt += f"Context: {ctx}\nQuestion: {q}\nReasoning: "

    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=1024).to(device)
    out = model.generate(**enc, max_new_tokens=150,
                         do_sample=False, num_beams=4,
                         no_repeat_ngram_size=4, repetition_penalty=1.05)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    # split reasoning vs answer
    reasoning, answer = decoded.split("Answer:", 1) if "Answer:" in decoded else ("", decoded)
    reasoning = reasoning.replace("Reasoning:", "").strip()
    return answer.strip(), reasoning, passages


print("🟢  Tuning‑1 answers (should refuse OOS & show CoT):\n")
for q in [
    "Who won the FIFA World Cup in 2010?",
    "What is the capital city of Australia?",
    "What is the function of the BRCA1 gene?",
]:
    ans, rationale, ctx = explain_question(q)
    print(f"Q: {q}\nA: {ans}\nReasoning: {rationale}\nTop passages:")
    for i, p in enumerate(ctx, 1):
        print(f"  [{i}] {p[:120]}…")
    print("-" * 60)


🟢  Tuning‑1 answers (should refuse OOS & show CoT):

Q: Who won the FIFA World Cup in 2010?
A: I’m a biomedical assistant and can’t answer that question.
Reasoning: OOS‑gate triggered
Top passages:
------------------------------------------------------------
Q: What is the capital city of Australia?
A: I’m a biomedical assistant and can’t answer that question.
Reasoning: OOS‑gate triggered
Top passages:
------------------------------------------------------------
Q: What is the function of the BRCA1 gene?
A: Our results demonstrate that the BRCA1 gene is responsible for apoptosis in BCR-ABL-positive leukemic cells.
Reasoning: 
Top passages:
  [1] nan…
  [2] The telomerase complex is responsible for telomere maintenance and represents a 
promising neoplasia therapeutic target.…
  [3] nan…
------------------------------------------------------------
